# Player Clustering Approach — Research Notes

## Features to Use (per-90 minutes)
- **Shots** — attacking threat
- **xG** — shot quality
- **Passes** — ball distribution
- **Pressures** — pressing intensity
- **Carries** — ball progression
- **Dribbles** — 1v1 ability
- **Interceptions** — defensive positioning
- **Blocks / Clearances** — defensive actions

## Normalization Strategy
Using **z-score standardization** (StandardScaler)
- More robust to outliers than min-max
- Better for K-Means which is sensitive to scale
- A player with unusually high shots won't skew the entire cluster

## Dimensionality Reduction
Using **PCA**
- For clustering: retain components explaining ≥85% variance (typically 4-5)
- For visualization: use first 2 components for 2D scatter plot
- Removes correlated features (e.g. passes and carries are related)

## Number of Clusters
- Target: **5-8 archetypes** (standard in football analytics)
- Selection method: elbow method + silhouette score
- Expected archetypes:
  - Pressing Forward
  - Creative Playmaker
  - Box-to-Box Midfielder
  - Defensive Anchor
  - Goal Poacher

## Minimum Minutes Filter
- Only include players with **≥ 270 minutes** played
- Ensures per-90 metrics are statistically meaningful
- Removes players with very few appearances

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

RAW = "bq_raw_statsbomb_sa_catalog.raw_statsbomb"

def parse_time_to_minutes(t):
    if t is None: return None
    s = str(t).strip()
    if s == "" or s.lower() == "nan": return None
    parts = s.split(":")
    try:
        if len(parts) == 2: return int(parts[0]) + int(parts[1]) / 60.0
        if len(parts) == 3: return int(parts[0]) * 60 + int(parts[1]) + int(parts[2]) / 60.0
    except (ValueError, TypeError): return None
    return None

parse_udf = F.udf(parse_time_to_minutes, DoubleType())

l = spark.table(f"{RAW}.lineups")
m = spark.table(f"{RAW}.matches")

position_df = (
    l.filter(F.col("player_id").isNotNull())
    .filter(F.col("position_name").isNotNull())
    .groupBy("player_id")
    .agg(F.first("position_name").alias("position_name"))
)

minutes_df = (
    l.join(m.select("match_id", "gender", "is_youth"), on="match_id", how="inner")
    .filter(F.col("player_name").isNotNull())
    .filter((F.col("gender") == "male") & (F.col("is_youth") == False))
    .withColumn("from_min", F.coalesce(parse_udf(F.col("from_time")), F.lit(0.0)))
    .withColumn("to_min", F.least(F.lit(120.0), F.coalesce(parse_udf(F.col("to_time")), F.lit(90.0))))
    .withColumn("minutes_played", F.greatest(F.lit(0.0), F.col("to_min") - F.col("from_min")))
    .groupBy("player_id", "player_name")
    .agg(
        F.sum("minutes_played").alias("total_minutes"),
        F.countDistinct("match_id").alias("matches_played")
    )
    .filter(F.col("total_minutes") >= 270)
    .join(position_df, on="player_id", how="left")
)

e = spark.table(f"{RAW}.events")

events_df = (
    e.join(m.select("match_id", "gender", "is_youth"), on="match_id", how="inner")
    .filter(F.col("player_id").isNotNull())
    .filter((F.col("gender") == "male") & (F.col("is_youth") == False))
    .groupBy("player_id")
    .agg(
        F.sum(F.when(F.col("type") == "Shot", F.col("shot_statsbomb_xg")).otherwise(0)).alias("xg"),
        F.sum(F.when(F.col("type") == "Shot", 1).otherwise(0)).alias("shots"),
        F.sum(F.when(F.col("type") == "Pass", 1).otherwise(0)).alias("passes"),
        F.sum(F.when((F.col("type") == "Pass") & (F.col("location_x") > 80), 1).otherwise(0)).alias("passes_att_third"),
        F.sum(F.when((F.col("type") == "Pass") & F.col("pass_outcome").isNull(), 1).otherwise(0)).alias("passes_completed"),
        F.sum(F.when(F.col("type") == "Pressure", 1).otherwise(0)).alias("pressures"),
        F.sum(F.when(F.col("type") == "Carry", 1).otherwise(0)).alias("carries"),
        F.sum(F.when(F.col("type") == "Dribble", 1).otherwise(0)).alias("dribbles"),
        F.sum(F.when(F.col("type") == "Interception", 1).otherwise(0)).alias("interceptions"),
        F.sum(F.when(F.col("type") == "Block", 1).otherwise(0)).alias("blocks"),
        F.sum(F.when(F.col("type") == "Clearance", 1).otherwise(0)).alias("clearances"),
        F.sum(F.when(F.col("type") == "Ball Recovery", 1).otherwise(0)).alias("ball_recoveries"),
        F.sum(F.when(F.col("type") == "Duel", 1).otherwise(0)).alias("duels"),
        F.sum(F.when(F.col("type") == "Foul Committed", 1).otherwise(0)).alias("fouls"),
    )
)

all_features = (
    minutes_df.join(events_df, on="player_id", how="inner")
    .withColumn("xg_p90",               F.col("xg")              / F.col("total_minutes") * 90)
    .withColumn("shots_p90",            F.col("shots")            / F.col("total_minutes") * 90)
    .withColumn("passes_p90",           F.col("passes")           / F.col("total_minutes") * 90)
    .withColumn("passes_att_third_p90", F.col("passes_att_third") / F.col("total_minutes") * 90)
    .withColumn("pressures_p90",        F.col("pressures")        / F.col("total_minutes") * 90)
    .withColumn("carries_p90",          F.col("carries")          / F.col("total_minutes") * 90)
    .withColumn("dribbles_p90",         F.col("dribbles")         / F.col("total_minutes") * 90)
    .withColumn("interceptions_p90",    F.col("interceptions")    / F.col("total_minutes") * 90)
    .withColumn("blocks_p90",           F.col("blocks")           / F.col("total_minutes") * 90)
    .withColumn("clearances_p90",       F.col("clearances")       / F.col("total_minutes") * 90)
    .withColumn("duels_p90",            F.col("duels")            / F.col("total_minutes") * 90)
    .withColumn("xg_per_shot",
        F.when(F.col("shots") > 0, F.col("xg") / F.col("shots")).otherwise(0))
    .withColumn("pass_completion_pct",
        F.when(F.col("passes") > 0, F.col("passes_completed") / F.col("passes") * 100).otherwise(None))
    .filter(
        (F.col("passes_p90") > 0) | (F.col("carries_p90") > 0) | (F.col("shots_p90") > 0)
    )
)

gk_df = all_features.filter(F.col("position_name") == "Goalkeeper")
outfield_df = all_features.filter(F.col("position_name") != "Goalkeeper")

print(f"Total players: {all_features.count()}")
print(f"Goalkeepers: {gk_df.count()}")
print(f"Outfield players: {outfield_df.count()}")

In [0]:
%python
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

feature_cols = [
    "xg_p90", "shots_p90", "passes_p90", "passes_att_third_p90",
    "pressures_p90", "carries_p90", "dribbles_p90", "interceptions_p90",
    "blocks_p90", "clearances_p90", "duels_p90", "xg_per_shot",
    "pass_completion_pct"
]

pdf_outfield = outfield_df.select(
    ["player_id", "player_name", "position_name", "total_minutes", "matches_played"] + feature_cols
).toPandas()

pdf_gk = gk_df.select(
    ["player_id", "player_name", "position_name", "total_minutes", "matches_played"] + feature_cols
).toPandas()
# N'Golo Kante has erroneous stats in raw StatsBomb data (passes_p90=448, carries_p90=360)
# This caused him to form his own cluster, so we remove him as a data quality outlier
pdf_outfield = pdf_outfield[pdf_outfield['player_name'] != "N'Golo Kanté"].reset_index(drop=True)
pdf_outfield[feature_cols] = pdf_outfield[feature_cols].fillna(pdf_outfield[feature_cols].median())

for col in feature_cols:
    if col in ['xg_p90', 'xg_per_shot']:
        continue
    cap = pdf_outfield[col].quantile(0.99)
    pdf_outfield[col] = pdf_outfield[col].clip(upper=cap)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pdf_outfield[feature_cols])

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

cumvar = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(10, 4))
plt.bar(range(1, len(cumvar)+1), pca.explained_variance_ratio_, alpha=0.6, label="Individual")
plt.plot(range(1, len(cumvar)+1), cumvar, marker='o', color='red', label="Cumulative")
plt.axhline(0.85, linestyle='--', color='gray', label="85% threshold")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("PCA Explained Variance (Outfield Players)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Outfield players: {len(pdf_outfield)}")
print(f"Components to reach 85% variance: {np.argmax(cumvar >= 0.85) + 1}")

In [0]:
%python
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_km = X_pca[:, :6]

inertias = []
silhouettes = []
K_range = range(3, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_km)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_km, labels, sample_size=2000, random_state=42))
    print(f"k={k} done")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(list(K_range), inertias, marker='o')
ax1.set_xlabel("k")
ax1.set_ylabel("Inertia")
ax1.set_title("Elbow Curve (Outfield Players)")
ax2.plot(list(K_range), silhouettes, marker='o', color='orange')
ax2.set_xlabel("k")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Scores (Outfield Players)")
plt.tight_layout()
plt.show()

In [0]:
%python
k = 5
km_final = KMeans(n_clusters=k, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_km)

colors = ['#1D9E75', '#BA7517', '#D4537E', '#7F77DD', '#999999']

plt.figure(figsize=(10, 7))
for i in range(k):
    mask = cluster_labels == i
    plt.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        s=10, alpha=0.4, color=colors[i], label=f'Cluster {i}'
    )
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-Means k=5 Clusters on PCA 2D (Outfield Players)")
plt.legend()
plt.tight_layout()
plt.show()

pdf_outfield['cluster'] = cluster_labels
print(pdf_outfield.groupby('cluster').size())

In [0]:
%python
cluster_means = pdf_outfield.groupby('cluster')[feature_cols].mean().round(2)
print(cluster_means.T)

In [0]:
%python
archetype_map = {
    0: "Low Activity",
    1: "Creative Winger",
    2: "Creative Playmaker",
    3: "Defensive Anchor",
    4: "Pressing Forward"
}

pdf_outfield['archetype'] = pdf_outfield['cluster'].map(archetype_map)
pdf_gk['archetype'] = 'Goalkeeper'
pdf_gk['cluster'] = -1

pdf_final = pd.concat([pdf_outfield, pdf_gk], ignore_index=True)
print(pdf_final.groupby('archetype').size())

In [0]:
%python
for archetype in pdf_final['archetype'].unique():
    top = (
        pdf_final[pdf_final['archetype'] == archetype]
        [['player_name', 'position_name', 'xg_p90', 'passes_p90', 'carries_p90']]
        .sort_values('xg_p90', ascending=False)
        .head(5)
    )
    print(f"\n{'='*50}")
    print(f"{archetype}")
    print(f"{'='*50}")
    print(top.to_string(index=False))

In [0]:
%python
import json
import os
import subprocess
import sys
import tempfile

PROJECT_ID = "football-capstone-mds-496219"
DATASET = "analytics"
TABLE = "cluster_assignments"

result_cols = ["player_id", "player_name", "archetype", "cluster"] + (["total_minutes", "matches_played"] if all(col in pdf_final.columns for col in ["total_minutes", "matches_played"]) else []) + feature_cols
export_df = pdf_final[result_cols].copy()

BQ_SERVICE_ACCOUNT_JSON_PATH = "/Volumes/workspace/default/gcp_keys/bq-sa.json"

def _gcp_credentials():
    json_path = (
        os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
        or os.environ.get("BQ_SERVICE_ACCOUNT_JSON_PATH")
        or BQ_SERVICE_ACCOUNT_JSON_PATH
    )
    if json_path and os.path.isfile(json_path):
        from google.oauth2 import service_account
        return service_account.Credentials.from_service_account_file(json_path)
    return None

def _ensure_google_cloud():
    for pkg in ("google-cloud-bigquery",):
        try:
            from google.cloud import bigquery
        except ImportError:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", pkg],
                stdout=subprocess.DEVNULL,
            )
            for _name in ("google.cloud.bigquery", "google.cloud", "google"):
                sys.modules.pop(_name, None)

_ensure_google_cloud()
from google.auth.exceptions import DefaultCredentialsError
from google.cloud import bigquery

creds = _gcp_credentials()
if creds is None:
    bq_client = bigquery.Client(project=PROJECT_ID)
else:
    bq_client = bigquery.Client(project=PROJECT_ID, credentials=creds)

table_id = f"{PROJECT_ID}.{DATASET}.{TABLE}"
load_job = bq_client.load_table_from_dataframe(
    export_df,
    table_id,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
load_job.result()
print(f"Exported {load_job.output_rows:,} rows to {table_id}")

In [0]:
%python
loadings_df = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)]
)
loadings_df.to_parquet('/tmp/pca_loadings.parquet')
print(loadings_df[['PC1', 'PC2', 'PC3']].round(3))

In [0]:
%python
import IPython.display as display
import base64

with open('/tmp/pca_loadings.parquet', 'rb') as f:
    data = f.read()

b64 = base64.b64encode(data).decode()
html = f'<a href="data:application/octet-stream;base64,{b64}" download="pca_loadings.parquet">Click here to download pca_loadings.parquet</a>'
display.display(display.HTML(html))

In [0]:
%python
import pandas as pd
df = pd.read_parquet("/tmp/pca_loadings.parquet")
print(df.shape)
print(df.columns.tolist())

In [0]:
%python
rename_map = {
    'xg_p90': 'xg_per_90',
    'shots_p90': 'shots_per_90',
    'passes_p90': 'passes_per_90',
    'passes_att_third_p90': 'passes_att_third_per_90',
    'pressures_p90': 'pressures_per_90',
    'carries_p90': 'carries_per_90',
    'dribbles_p90': 'dribbles_per_90',
    'interceptions_p90': 'interceptions_per_90',
    'blocks_p90': 'blocks_per_90',
    'clearances_p90': 'clearances_per_90',
    'duels_p90': 'duels_per_90',
    'xg_per_shot': 'xg_per_shot',
    'pass_completion_pct': 'pass_completion_pct'
}

loadings_df = pd.DataFrame(
    pca.components_.T,
    index=[rename_map.get(f, f) for f in feature_cols],
    columns=[f'PC{i+1}' for i in range(pca.n_components_)]
).reset_index()
loadings_df.rename(columns={'index': 'feature'}, inplace=True)

table_id = "football-capstone-mds-496219.analytics.pca_loadings"
load_job = bq_client.load_table_from_dataframe(
    loadings_df,
    table_id,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
load_job.result()
print(f"Exported {len(loadings_df)} features to {table_id}")
print(loadings_df[['feature', 'PC1']].to_string(index=False))

In [0]:
%python
print(feature_cols)